# Outage Duration Uncertainty — Show & Tell

**Plant Millbrook · Refuelling Outage OH-005**

This notebook walks through the full pipeline:

| Section | What it shows |
|---------|---------------|
| 1 | Text pre-processing — abbreviation expansion, spell correction |
| 2 | Pre-processing quality benchmark — clean vs contaminated similarity |
| 3 | Duration estimation — analogue retrieval and distribution fitting |
| 4 | Schedule risk — Monte Carlo propagation and critical-path analysis |

No external servers required — all steps run offline against synthetic historical data.

In [ ]:
import sys
from pathlib import Path

# Make the package importable when running from the demos/ folder
_ROOT = Path(".").resolve().parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import math
import csv
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

print("Environment ready.")

---
## Section 1 · Text Pre-processing

P6 activity descriptions are noisy: they contain abbreviations (`MOV`, `RHR`), typos (`presure`, `transmiter`), mixed case, telegraphic style, and embedded component tags.

The pre-processing pipeline runs four stages in sequence:

```
raw description
  → ComponentIdRemover   (strip tags like PT-455A)
  → TextacyPreprocessor  (normalise unicode, whitespace)
  → AbbreviationResolver (MOV → motor operated valve)
  → DomainSpellChecker   (presure → pressure)
  → cleaned description
```

In [ ]:
from outage_uncertainty.preprocessing.abbreviations import AbbreviationResolver
from outage_uncertainty.preprocessing.spell_checker import DomainSpellChecker
from outage_uncertainty.preprocessing.cleaners import (
    ActivityCleaner, ComponentIdRemover, TextacyPreprocessor, IdentityTransform,
)
from outage_uncertainty.domain.activity import ActivityCase

# Build the cleaning pipeline (no Excel file needed — nuclear supplement is built-in)
cleaner = ActivityCleaner(
    component_id_remover=ComponentIdRemover(),
    preprocessor=TextacyPreprocessor(),
    abbreviation_expander=AbbreviationResolver(),
    spell_checker=DomainSpellChecker(cutoff=0.85),
)

print("Cleaner pipeline instantiated.")

In [ ]:
# ── Load benchmark data ───────────────────────────────────────────────────
# CSVs live in the sibling unexpected_act_workflow_1/ folder.
# _ROOT is set in Cell 1 as Path(".").resolve().parent (= demos/).
_WF1_DIR = _ROOT / "unexpected_act_workflow_1"
benchmark_rows = []
with open(_WF1_DIR / "outage_cleaning_benchmark.csv", newline="") as f:
    for row in csv.DictReader(f):
        benchmark_rows.append(row)

print(f"Loaded {len(benchmark_rows)} benchmark rows across "
      f"{len({r['category'] for r in benchmark_rows})} categories")

### 1a · Step-by-step trace on a single description

In [ ]:
# Pick an illustrative example
example = benchmark_rows[1]   # OD002 — heavy contamination
raw = example["contaminated_description"]
clean_ref = example["clean_description"]

# Run each stage manually to show intermediate state
_resolver = AbbreviationResolver()
_spell    = DomainSpellChecker(cutoff=0.85)
_remover  = ComponentIdRemover()
_preproc  = TextacyPreprocessor()

stage1 = _remover.transform(raw)
stage2 = _preproc.transform(stage1)
stage3 = _resolver.transform(stage2)
stage4 = _spell.transform(stage3)

stages = [
    ("Raw (P6 input)",         raw),
    ("After ComponentIdRemover", stage1),
    ("After TextacyPreprocessor", stage2),
    ("After AbbreviationResolver", stage3),
    ("After DomainSpellChecker",   stage4),
    ("Reference (clean)",      clean_ref),
]

print(f"Contamination profile: {example['contamination_profile']}\n")
for label, text in stages:
    marker = "✓" if label.startswith("Reference") else " "
    print(f"{marker} [{label}]")
    print(f"    {text}")
    print()

### 1b · Batch cleaning — spot-check 10 examples

In [ ]:
import random
random.seed(42)
sample_rows = random.sample(benchmark_rows, 10)

def _clean(raw_text: str) -> str:
    act = ActivityCase(
        activity_id="X", outage_id="O", plant_id="P",
        raw_description=raw_text,
    )
    return cleaner.clean(act).cleaned_description or ""

fig, axes = plt.subplots(10, 1, figsize=(14, 8))
fig.suptitle("Pre-processing: contaminated → cleaned", fontsize=12, fontweight="bold", y=1.01)

colors = {"valve_maintenance": "#4C72B0", "pump_maintenance": "#DD8452",
          "instrument_calibration": "#55A868", "electrical_maintenance": "#C44E52",
          "major_component_work": "#8172B2", "heat_exchanger_work": "#937860",
          "testing_and_restoration": "#DA8BC3", "access_support": "#8C8C8C",
          "chemistry_and_flushing": "#CCB974", "supports_and_snubbers": "#64B5CD"}

for ax, row in zip(axes, sample_rows):
    cat   = row["category"]
    raw   = row["contaminated_description"]
    cleaned = _clean(raw)
    color = colors.get(cat, "#999999")

    ax.barh([0], [1], color=color, alpha=0.15, height=0.9)
    ax.text(0.01, 0.65, f"IN:  {raw[:95]}",
            transform=ax.transAxes, fontsize=7.5, color="#555555",
            va="center", fontfamily="monospace")
    ax.text(0.01, 0.20, f"OUT: {cleaned[:95]}",
            transform=ax.transAxes, fontsize=7.5, color="#1a1a1a",
            va="center", fontfamily="monospace", fontweight="bold")
    ax.text(0.97, 0.5, cat.replace("_", " "),
            transform=ax.transAxes, fontsize=7, color=color,
            ha="right", va="center")
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, 1.5)
    ax.axis("off")

plt.tight_layout()
plt.show()

---
## Section 2 · Pre-processing Quality Benchmark

We quantify how much the cleaner improves text quality using two metrics:

- **Normalised Edit Distance** — character-level similarity between cleaned output and the reference (lower = better recovery)
- **Token Overlap (F1)** — word-level F1 between cleaned output and reference (higher = better)

In [ ]:
from difflib import SequenceMatcher
from collections import defaultdict

def _ned(a: str, b: str) -> float:
    """Normalised edit distance: 1 - SequenceMatcher ratio."""
    return 1.0 - SequenceMatcher(None, a.lower(), b.lower()).ratio()

def _token_f1(pred: str, ref: str) -> float:
    p_tokens = set(pred.lower().split())
    r_tokens = set(ref.lower().split())
    if not p_tokens or not r_tokens:
        return 0.0
    tp = len(p_tokens & r_tokens)
    precision = tp / len(p_tokens)
    recall    = tp / len(r_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

# Compute per-category metrics: before and after cleaning
cat_before_ned  = defaultdict(list)
cat_after_ned   = defaultdict(list)
cat_before_f1   = defaultdict(list)
cat_after_f1    = defaultdict(list)

for row in benchmark_rows:
    cat     = row["category"]
    raw     = row["contaminated_description"]
    ref     = row["clean_description"]
    cleaned = _clean(raw)

    cat_before_ned[cat].append(_ned(raw, ref))
    cat_after_ned[cat].append(_ned(cleaned, ref))
    cat_before_f1[cat].append(_token_f1(raw, ref))
    cat_after_f1[cat].append(_token_f1(cleaned, ref))

categories = sorted(cat_before_ned.keys())
before_ned = [sum(cat_before_ned[c]) / len(cat_before_ned[c]) for c in categories]
after_ned  = [sum(cat_after_ned[c])  / len(cat_after_ned[c])  for c in categories]
before_f1  = [sum(cat_before_f1[c])  / len(cat_before_f1[c])  for c in categories]
after_f1   = [sum(cat_after_f1[c])   / len(cat_after_f1[c])   for c in categories]

# Overall
all_before_ned = [v for vals in cat_before_ned.values() for v in vals]
all_after_ned  = [v for vals in cat_after_ned.values()  for v in vals]
all_before_f1  = [v for vals in cat_before_f1.values()  for v in vals]
all_after_f1   = [v for vals in cat_after_f1.values()   for v in vals]

print(f"Overall NED   — before: {sum(all_before_ned)/len(all_before_ned):.3f}  "
      f"after: {sum(all_after_ned)/len(all_after_ned):.3f}")
print(f"Overall Token F1 — before: {sum(all_before_f1)/len(all_before_f1):.3f}  "
      f"after: {sum(all_after_f1)/len(all_after_f1):.3f}")

In [ ]:
cat_labels = [c.replace("_", "\n") for c in categories]
x = np.arange(len(categories))
w = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

# ── NED (lower is better) ─────────────────────────────────────────────────
ax1.bar(x - w/2, before_ned, w, label="Before cleaning",
        color="#C44E52", alpha=0.85)
ax1.bar(x + w/2, after_ned,  w, label="After cleaning",
        color="#55A868", alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(cat_labels, fontsize=7.5)
ax1.set_ylabel("Normalised Edit Distance (lower = better)")
ax1.set_title("Text Recovery — Edit Distance")
ax1.legend(fontsize=9)
ax1.set_ylim(0, 1)

# ── Token F1 (higher is better) ───────────────────────────────────────────
ax2.bar(x - w/2, before_f1, w, label="Before cleaning",
        color="#C44E52", alpha=0.85)
ax2.bar(x + w/2, after_f1,  w, label="After cleaning",
        color="#55A868", alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(cat_labels, fontsize=7.5)
ax2.set_ylabel("Token F1 (higher = better)")
ax2.set_title("Text Recovery — Word Overlap")
ax2.legend(fontsize=9)
ax2.set_ylim(0, 1)

fig.suptitle("Pre-processing benchmark: 175 contaminated descriptions",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 3 · Duration Estimation

For each planned activity the service:
1. Cleans the description
2. Retrieves the most similar historical activities (lexical + semantic + context scoring)
3. Separates routine vs extended durations (IQR outlier detection)
4. Fits a distribution to the routine pool
5. Returns a `DurationDistribution` with P50 / P80 / P90 percentiles and a confidence tier

In [ ]:
from outage_uncertainty.api.facade import build_duration_uncertainty_service

service = build_duration_uncertainty_service()
print("Service ready.")

In [ ]:
# ── Historical database (3 prior outages, 35 activities) ─────────────────
# Adapted from activity_duration_demo.py — Plant Millbrook, fictional PWR

HISTORICAL_ROWS = [
    # OH-001  2022 Unit 1
    {"activity_id":"H001","outage_id":"OH-001","plant_id":"Millbrook-U1",
     "raw_description":"Main coolant pump MCP-1A mechanical seal replacement",
     "discipline":"mechanical","task_family":"replacement","component_family":"pump",
     "planned_duration_hours":20.0,"actual_duration_hours":23.5,
     "has_rp_hold":True,"outage_phase":"planned outage"},
    {"activity_id":"H002","outage_id":"OH-001","plant_id":"Millbrook-U1",
     "raw_description":"RHR pump 1A discharge valve packing replacement",
     "discipline":"mechanical","task_family":"replacement","component_family":"valve",
     "planned_duration_hours":6.0,"actual_duration_hours":7.5,
     "outage_phase":"planned outage"},
    {"activity_id":"H003","outage_id":"OH-001","plant_id":"Millbrook-U1",
     "raw_description":"MSIV valve stem inspection unit 1 A train",
     "discipline":"mechanical","task_family":"inspection","component_family":"valve",
     "planned_duration_hours":10.0,"actual_duration_hours":11.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H004","outage_id":"OH-001","plant_id":"Millbrook-U1",
     "raw_description":"Charging pump 1B complete overhaul impeller and bearings",
     "discipline":"mechanical","task_family":"refurbishment","component_family":"pump",
     "planned_duration_hours":40.0,"actual_duration_hours":47.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H005","outage_id":"OH-001","plant_id":"Millbrook-U1",
     "raw_description":"Level transmitter LT-101 channel calibration loop check",
     "discipline":"I&C","task_family":"calibration","component_family":"instrument",
     "planned_duration_hours":3.0,"actual_duration_hours":3.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H006","outage_id":"OH-001","plant_id":"Millbrook-U1",
     "raw_description":"Flow transmitter FT-203 calibration set point verification",
     "discipline":"I&C","task_family":"calibration","component_family":"instrument",
     "planned_duration_hours":3.0,"actual_duration_hours":4.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H007","outage_id":"OH-001","plant_id":"Millbrook-U1",
     "raw_description":"ECCS injection valve MOV actuator replacement",
     "discipline":"mechanical","task_family":"replacement","component_family":"valve",
     "planned_duration_hours":24.0,"actual_duration_hours":38.0,
     "has_rp_hold":True,"has_clearance":True,"outage_phase":"planned outage"},
    {"activity_id":"H008","outage_id":"OH-001","plant_id":"Millbrook-U1",
     "raw_description":"Main transformer A bushing inspection and oil sampling",
     "discipline":"electrical","task_family":"inspection","component_family":"transformer",
     "planned_duration_hours":6.0,"actual_duration_hours":6.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H009","outage_id":"OH-001","plant_id":"Millbrook-U1",
     "raw_description":"Bus 2A medium voltage breaker preventive maintenance",
     "discipline":"electrical","task_family":"maintenance","component_family":"breaker",
     "planned_duration_hours":4.0,"actual_duration_hours":4.5,
     "outage_phase":"planned outage"},
    {"activity_id":"H010","outage_id":"OH-001","plant_id":"Millbrook-U1",
     "raw_description":"Condenser tube bundle cleaning and inspection",
     "discipline":"mechanical","task_family":"cleaning","component_family":"condenser",
     "planned_duration_hours":12.0,"actual_duration_hours":14.0,
     "outage_phase":"planned outage"},
    # OH-002  2023 Unit 1
    {"activity_id":"H011","outage_id":"OH-002","plant_id":"Millbrook-U1",
     "raw_description":"Main coolant pump MCP-1B mechanical seal replacement",
     "discipline":"mechanical","task_family":"replacement","component_family":"pump",
     "planned_duration_hours":20.0,"actual_duration_hours":26.0,
     "has_rp_hold":True,"outage_phase":"planned outage"},
    {"activity_id":"H012","outage_id":"OH-002","plant_id":"Millbrook-U1",
     "raw_description":"RHR pump 1B suction valve packing replacement",
     "discipline":"mechanical","task_family":"replacement","component_family":"valve",
     "planned_duration_hours":6.0,"actual_duration_hours":6.5,
     "outage_phase":"planned outage"},
    {"activity_id":"H013","outage_id":"OH-002","plant_id":"Millbrook-U1",
     "raw_description":"MSIV valve B train full inspection disassembly",
     "discipline":"mechanical","task_family":"inspection","component_family":"valve",
     "planned_duration_hours":10.0,"actual_duration_hours":13.5,
     "outage_phase":"planned outage"},
    {"activity_id":"H014","outage_id":"OH-002","plant_id":"Millbrook-U1",
     "raw_description":"Charging pump 1A overhaul casing and mechanical seal",
     "discipline":"mechanical","task_family":"refurbishment","component_family":"pump",
     "planned_duration_hours":40.0,"actual_duration_hours":53.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H015","outage_id":"OH-002","plant_id":"Millbrook-U1",
     "raw_description":"Pressure transmitter PT-305 calibration set point",
     "discipline":"I&C","task_family":"calibration","component_family":"instrument",
     "planned_duration_hours":3.0,"actual_duration_hours":3.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H016","outage_id":"OH-002","plant_id":"Millbrook-U1",
     "raw_description":"Temperature element TE-401 replacement loop verification",
     "discipline":"I&C","task_family":"replacement","component_family":"instrument",
     "planned_duration_hours":4.0,"actual_duration_hours":5.5,
     "outage_phase":"planned outage"},
    {"activity_id":"H017","outage_id":"OH-002","plant_id":"Millbrook-U1",
     "raw_description":"HPSI pump 1A mechanical seal and bearing overhaul",
     "discipline":"mechanical","task_family":"refurbishment","component_family":"pump",
     "planned_duration_hours":32.0,"actual_duration_hours":35.0,
     "has_rp_hold":True,"outage_phase":"planned outage"},
    {"activity_id":"H018","outage_id":"OH-002","plant_id":"Millbrook-U1",
     "raw_description":"EDG emergency diesel generator fuel oil system maintenance",
     "discipline":"mechanical","task_family":"maintenance","component_family":"generator",
     "planned_duration_hours":16.0,"actual_duration_hours":16.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H019","outage_id":"OH-002","plant_id":"Millbrook-U1",
     "raw_description":"CCW heat exchanger tube side cleaning and inspection",
     "discipline":"mechanical","task_family":"cleaning","component_family":"heat exchanger",
     "planned_duration_hours":10.0,"actual_duration_hours":12.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H020","outage_id":"OH-002","plant_id":"Millbrook-U1",
     "raw_description":"Switchgear bus A annual preventive maintenance",
     "discipline":"electrical","task_family":"maintenance","component_family":"switchgear",
     "planned_duration_hours":6.0,"actual_duration_hours":6.5,
     "outage_phase":"planned outage"},
    # OH-003  2024 Unit 2
    {"activity_id":"H021","outage_id":"OH-003","plant_id":"Millbrook-U2",
     "raw_description":"Main coolant pump MCP-2A mechanical seal complete replacement",
     "discipline":"mechanical","task_family":"replacement","component_family":"pump",
     "planned_duration_hours":20.0,"actual_duration_hours":22.0,
     "has_rp_hold":True,"outage_phase":"planned outage"},
    {"activity_id":"H022","outage_id":"OH-003","plant_id":"Millbrook-U2",
     "raw_description":"Pressurizer heater replacement bundle 2A",
     "discipline":"electrical","task_family":"replacement","component_family":"heater",
     "planned_duration_hours":14.0,"actual_duration_hours":18.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H023","outage_id":"OH-003","plant_id":"Millbrook-U2",
     "raw_description":"MSIV valve A unit 2 full disassembly inspection and reassembly",
     "discipline":"mechanical","task_family":"inspection","component_family":"valve",
     "planned_duration_hours":10.0,"actual_duration_hours":10.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H024","outage_id":"OH-003","plant_id":"Millbrook-U2",
     "raw_description":"Charging pump 2B overhaul impeller bearings casing",
     "discipline":"mechanical","task_family":"refurbishment","component_family":"pump",
     "planned_duration_hours":40.0,"actual_duration_hours":44.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H025","outage_id":"OH-003","plant_id":"Millbrook-U2",
     "raw_description":"Differential pressure transmitter DP-201 calibration",
     "discipline":"I&C","task_family":"calibration","component_family":"instrument",
     "planned_duration_hours":3.0,"actual_duration_hours":3.5,
     "outage_phase":"planned outage"},
    {"activity_id":"H026","outage_id":"OH-003","plant_id":"Millbrook-U2",
     "raw_description":"Safety injection pump 2B mechanical seal replacement",
     "discipline":"mechanical","task_family":"replacement","component_family":"pump",
     "planned_duration_hours":18.0,"actual_duration_hours":21.0,
     "has_rp_hold":True,"outage_phase":"planned outage"},
    {"activity_id":"H027","outage_id":"OH-003","plant_id":"Millbrook-U2",
     "raw_description":"RHR heat exchanger tube bundle inspection eddy current",
     "discipline":"mechanical","task_family":"inspection","component_family":"heat exchanger",
     "planned_duration_hours":24.0,"actual_duration_hours":28.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H028","outage_id":"OH-003","plant_id":"Millbrook-U2",
     "raw_description":"Bus 4B medium voltage breaker overhaul and testing",
     "discipline":"electrical","task_family":"refurbishment","component_family":"breaker",
     "planned_duration_hours":8.0,"actual_duration_hours":9.5,
     "outage_phase":"planned outage"},
    {"activity_id":"H029","outage_id":"OH-003","plant_id":"Millbrook-U2",
     "raw_description":"Service water pump SWP-2A bearing replacement",
     "discipline":"mechanical","task_family":"replacement","component_family":"pump",
     "planned_duration_hours":8.0,"actual_duration_hours":9.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H030","outage_id":"OH-003","plant_id":"Millbrook-U2",
     "raw_description":"Main feedwater regulating valve actuator replacement",
     "discipline":"mechanical","task_family":"replacement","component_family":"valve",
     "planned_duration_hours":16.0,"actual_duration_hours":19.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H031","outage_id":"OH-003","plant_id":"Millbrook-U2",
     "raw_description":"Containment spray pump CSP-2A mechanical seal overhaul",
     "discipline":"mechanical","task_family":"refurbishment","component_family":"pump",
     "planned_duration_hours":20.0,"actual_duration_hours":24.0,
     "has_rp_hold":True,"outage_phase":"planned outage"},
    {"activity_id":"H032","outage_id":"OH-003","plant_id":"Millbrook-U2",
     "raw_description":"Moisture separator reheater MSR inspection and tube plugging",
     "discipline":"mechanical","task_family":"inspection","component_family":"heat exchanger",
     "planned_duration_hours":30.0,"actual_duration_hours":36.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H033","outage_id":"OH-003","plant_id":"Millbrook-U2",
     "raw_description":"125V DC battery charger replacement unit 2A",
     "discipline":"electrical","task_family":"replacement","component_family":"battery",
     "planned_duration_hours":6.0,"actual_duration_hours":7.0,
     "outage_phase":"planned outage"},
    {"activity_id":"H034","outage_id":"OH-003","plant_id":"Millbrook-U2",
     "raw_description":"CRD mechanism housing weld inspection ultrasonic",
     "discipline":"mechanical","task_family":"inspection","component_family":"crd",
     "planned_duration_hours":48.0,"actual_duration_hours":52.0,
     "has_rp_hold":True,"outage_phase":"planned outage"},
    {"activity_id":"H035","outage_id":"OH-003","plant_id":"Millbrook-U2",
     "raw_description":"Turbine governor valve GV-1 overhaul and stroke test",
     "discipline":"mechanical","task_family":"refurbishment","component_family":"valve",
     "planned_duration_hours":24.0,"actual_duration_hours":27.0,
     "outage_phase":"planned outage"},
]

print(f"Historical database: {len(HISTORICAL_ROWS)} activities from 3 outages")

In [ ]:
# ── Planned activities for OH-005 ─────────────────────────────────────────
PLANNED = [
    {"activity_id": "Q-MCP-2B",   "outage_id": "OH-005", "plant_id": "Millbrook-U2",
     "raw_description": "MCP-2B mechanical seal and bearing replacement",
     "discipline": "mechanical", "component_family": "pump",
     "planned_duration_hours": 20.0, "has_rp_hold": True,
     "outage_phase": "planned outage"},

    {"activity_id": "Q-MSIV-2",   "outage_id": "OH-005", "plant_id": "Millbrook-U2",
     "raw_description": "MSIV valve unit 2 full disassembly and inspection",
     "discipline": "mechanical", "component_family": "valve",
     "planned_duration_hours": 10.0,
     "outage_phase": "planned outage"},

    {"activity_id": "Q-LT-CALIB", "outage_id": "OH-005", "plant_id": "Millbrook-U2",
     "raw_description": "Level transmitter LT-220 calibration and loop check",
     "discipline": "I&C", "component_family": "instrument",
     "planned_duration_hours": 3.0,
     "outage_phase": "planned outage"},

    {"activity_id": "Q-HX-CCW",   "outage_id": "OH-005", "plant_id": "Millbrook-U2",
     "raw_description": "CCW heat exchanger tube side inspection and cleaning",
     "discipline": "mechanical", "component_family": "heat exchanger",
     "planned_duration_hours": 10.0,
     "outage_phase": "planned outage"},

    {"activity_id": "Q-BKR-4A",   "outage_id": "OH-005", "plant_id": "Millbrook-U2",
     "raw_description": "Bus 4A medium voltage breaker overhaul",
     "discipline": "electrical", "component_family": "breaker",
     "planned_duration_hours": 8.0,
     "outage_phase": "planned outage"},

    {"activity_id": "Q-DCS-UPGRDE","outage_id": "OH-005", "plant_id": "Millbrook-U2",
     "raw_description": "DCS digital control system upgrade and functional test",
     "discipline": "I&C", "component_family": "controller",
     "planned_duration_hours": 40.0,
     "outage_phase": "planned outage"},
]

print(f"Planned activities for OH-005: {len(PLANNED)}")

In [ ]:
# ── Run estimation ────────────────────────────────────────────────────────
results = []
for act in PLANNED:
    est = service.estimate_activity(
        query_row=act,
        historical_rows=HISTORICAL_ROWS,
    )
    results.append((act, est))
    print(f"  {act['activity_id']:<20}  tier={est.confidence_tier:<6}  "
          f"type={est.uncertainty_type:<12}  support={est.support_count}")

### 3a · Duration estimate summary

In [ ]:
act_ids    = [a["activity_id"] for a, _ in results]
planned_h  = [a["planned_duration_hours"] for a, _ in results]
p50        = [e.estimated_distribution.p50 for _, e in results]
p80        = [e.estimated_distribution.p80 for _, e in results]
p90        = [e.estimated_distribution.p90 for _, e in results]
tiers      = [e.confidence_tier for _, e in results]
utypes     = [e.uncertainty_type for _, e in results]

tier_color = {"HIGH": "#2ecc71", "MEDIUM": "#f39c12", "LOW": "#e74c3c"}
colors_bars = [tier_color.get(t, "#aaaaaa") for t in tiers]

x = np.arange(len(act_ids))
fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(x, planned_h, 0.5, label="Planned", color="#d0d0d0", zorder=2)
ax.bar(x + 0.0, p50, 0.5, label="P50 estimate", color=colors_bars, alpha=0.85, zorder=3)

# P80 and P90 error caps
for i, (lo, hi) in enumerate(zip(p80, p90)):
    ax.plot([x[i], x[i]], [lo, hi], color="#2c3e50", lw=2, zorder=4)
    ax.plot([x[i] - 0.12, x[i] + 0.12], [lo, lo], color="#2c3e50", lw=1.5, zorder=4)
    ax.plot([x[i] - 0.12, x[i] + 0.12], [hi, hi], color="#2c3e50", lw=1.5, zorder=4)

# Tier labels
for i, (tier, utype) in enumerate(zip(tiers, utypes)):
    ax.text(x[i], p90[i] + 0.8, tier, ha="center", va="bottom",
            fontsize=8, fontweight="bold", color=tier_color.get(tier, "#555"))

ax.set_xticks(x)
ax.set_xticklabels(act_ids, rotation=15, ha="right")
ax.set_ylabel("Duration (hours)")
ax.set_title("OH-005 Duration Estimates — P50 bars, P80–P90 whiskers, confidence tier above",
             fontsize=11)

# Legend patches
patches = [mpatches.Patch(color=v, label=f"{k} confidence") for k, v in tier_color.items()]
patches.append(mpatches.Patch(color="#d0d0d0", label="Planned duration"))
ax.legend(handles=patches, fontsize=9, loc="upper left")

ax.yaxis.grid(True, alpha=0.3, zorder=0)
plt.tight_layout()
plt.show()

### 3b · Top-3 analogues for the MCP seal replacement

In [ ]:
# Analogue deep-dive for Q-MCP-2B (well-supported) and Q-DCS-UPGRDE (epistemic)
_SEP = "═" * 62
for target_id in ("Q-MCP-2B", "Q-DCS-UPGRDE"):
    for act, est in results:
        if act["activity_id"] != target_id:
            continue

        print(f"\n{_SEP}")
        print(f"  {target_id}  —  {act['raw_description']}")
        print(f"  Confidence tier : {est.confidence_tier}")
        print(f"  Uncertainty type: {est.uncertainty_type}")
        print(f"  Analogues used  : {est.support_count}")
        d = est.estimated_distribution
        print(f"  P50={d.p50:.1f}h   P80={d.p80:.1f}h   P90={d.p90:.1f}h")

        if est.matched_cases:
            print("\n  Top analogues:")
            print(f"  {'ID':<8} {'Score':>6}  {'Description':<45} {'Actual':>7}")
            print(f"  {'-'*8} {'-'*6}  {'-'*45} {'-'*7}")
            hist_by_id = {r["activity_id"]: r for r in HISTORICAL_ROWS}
            top3 = sorted(est.matched_cases, key=lambda m: m.total_score, reverse=True)[:3]
            for m in top3:
                h = hist_by_id.get(m.candidate_activity_id, {})
                desc = h.get("raw_description", "?")[:44]
                actual = h.get("actual_duration_hours", 0.0)
                print(f"  {m.candidate_activity_id:<8} {m.total_score:>6.3f}  {desc:<45} {actual:>6.1f}h")
        if est.warnings:
            print("\n  ⚠ Warnings:")
            for w in est.warnings:
                print(f"    {w}")


### 3c · Duration distribution — routine vs extended

In [ ]:
# Show distribution histograms for activities that have enough samples
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for ax, (act, est) in zip(axes, results):
    d = est.estimated_distribution
    samples = d.samples or []
    planned = act["planned_duration_hours"]
    color   = tier_color.get(est.confidence_tier, "#888")

    if len(samples) >= 3:
        ax.hist(samples, bins=min(15, max(5, len(samples)//2)),
                color=color, alpha=0.75, edgecolor="white", linewidth=0.5)
        ax.axvline(d.p50, color="#2c3e50",  lw=2,   ls="-",  label=f"P50 {d.p50:.1f}h")
        ax.axvline(d.p80, color="#e67e22",  lw=1.5, ls="--", label=f"P80 {d.p80:.1f}h")
        ax.axvline(d.p90, color="#e74c3c",  lw=1.5, ls=":",  label=f"P90 {d.p90:.1f}h")
        ax.axvline(planned, color="#7f8c8d", lw=1, ls="-.", label=f"Plan {planned:.0f}h")
        ax.legend(fontsize=7.5)
    else:
        # Fallback: point estimate
        ax.bar([d.p50], [1], width=1.5, color=color, alpha=0.7)
        ax.text(d.p50, 0.5, f"P50={d.p50:.1f}h\n(fallback)",
                ha="center", va="center", fontsize=9)
        ax.set_ylim(0, 1.5)

    ax.set_title(f"{act['activity_id']}\n{est.confidence_tier} · {est.uncertainty_type}",
                 fontsize=9)
    ax.set_xlabel("Duration (h)", fontsize=8)
    ax.set_ylabel("Count", fontsize=8)

fig.suptitle("Routine duration distributions — OH-005 planned activities",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 4 · Monte Carlo Schedule Risk

The 6 planned activities are embedded in a 10-node schedule network for OH-005.  
A 2,000-iteration Monte Carlo simulation samples each activity's duration from its estimated distribution and propagates uncertainty through to the outage finish time via the Critical Path Method.

```
Q-INIT ──┬──► Q-MCP-2B ──► Q-REPL ──────────────────────────► Q-END
         ├──► Q-MSIV-2 ──────────────────────────────────────► Q-END
         ├──► Q-HX-CCW ──► Q-LT-CALIB ──► Q-BKR-4A ─────────► Q-END
         └──► Q-DCS-UPGRDE ──────────────────────────────────► Q-END
```

In [ ]:
from outage_uncertainty.domain.schedule import ScheduleActivity
from outage_uncertainty.domain.duration import DurationDistribution
from outage_uncertainty.schedule_risk.schedule_graph import ScheduleNetwork
from outage_uncertainty.schedule_risk.scenario_runner import ScenarioRunner

# Build DurationDistribution objects from the estimates
est_by_id = {a["activity_id"]: e for a, e in results}

def _dist(act_id):
    est = est_by_id.get(act_id)
    return est.estimated_distribution if est else None

# Network activities — 10 nodes
activities = [
    ScheduleActivity("Q-INIT",      "Outage start / tag-outs",
                     successors=["Q-MCP-2B","Q-MSIV-2","Q-HX-CCW","Q-DCS-UPGRDE"],
                     baseline_duration_hours=4.0),
    ScheduleActivity("Q-MCP-2B",    "MCP-2B seal replacement",
                     predecessors=["Q-INIT"], successors=["Q-REPL"],
                     baseline_duration_hours=20.0,
                     duration_distribution=_dist("Q-MCP-2B")),
    ScheduleActivity("Q-MSIV-2",    "MSIV unit 2 inspection",
                     predecessors=["Q-INIT"], successors=["Q-END"],
                     baseline_duration_hours=10.0,
                     duration_distribution=_dist("Q-MSIV-2")),
    ScheduleActivity("Q-HX-CCW",    "CCW HX clean & inspect",
                     predecessors=["Q-INIT"], successors=["Q-LT-CALIB"],
                     baseline_duration_hours=10.0,
                     duration_distribution=_dist("Q-HX-CCW")),
    ScheduleActivity("Q-LT-CALIB",  "LT-220 calibration",
                     predecessors=["Q-HX-CCW"], successors=["Q-BKR-4A"],
                     baseline_duration_hours=3.0,
                     duration_distribution=_dist("Q-LT-CALIB")),
    ScheduleActivity("Q-BKR-4A",    "Bus 4A breaker overhaul",
                     predecessors=["Q-LT-CALIB"], successors=["Q-END"],
                     baseline_duration_hours=8.0,
                     duration_distribution=_dist("Q-BKR-4A")),
    ScheduleActivity("Q-DCS-UPGRDE","DCS upgrade & test",
                     predecessors=["Q-INIT"], successors=["Q-END"],
                     baseline_duration_hours=40.0,
                     duration_distribution=_dist("Q-DCS-UPGRDE")),
    ScheduleActivity("Q-REPL",      "Reactor plant re-entry checks",
                     predecessors=["Q-MCP-2B"], successors=["Q-END"],
                     baseline_duration_hours=6.0),
    ScheduleActivity("Q-END",       "Outage end / restart",
                     predecessors=["Q-REPL","Q-MSIV-2","Q-BKR-4A","Q-DCS-UPGRDE"],
                     baseline_duration_hours=2.0),
]

network = ScheduleNetwork(activities)

# Baseline critical path (using baseline durations)
baseline_durations = {a.activity_id: a.baseline_duration_hours for a in activities}
baseline    = network.compute_critical_path(baseline_durations)
baseline_cp = baseline["cp_time"]
print(f"Baseline critical path : {' -> '.join(baseline['cp_path'])}")
print(f"Baseline CP duration   : {baseline_cp:.1f} h")


In [ ]:
# ── Monte Carlo ──────────────────────────────────────────────────────────
runner = ScenarioRunner()
risk   = runner.run(network, baseline_cp_time=baseline_cp, n_samples=2000)

# The ScenarioRunner returns the cp_analyzer result dict.
# cp_times live on the SimulationResult; rebuild from the analyzer output.
# Keys: p50_finish, p80_finish, p90_finish, robustness, criticality_index,
#       expected_drag, cp_sensitivity, expected_delay, schedule_std_dev

overrun_prob = 1.0 - risk.get("robustness", 0.0)

print(f"Simulations run        : 2 000")
print(f"Baseline CP duration   : {baseline_cp:.1f} h")
print(f"P50 outage duration    : {risk['p50_finish']:.1f} h")
print(f"P80 outage duration    : {risk['p80_finish']:.1f} h")
print(f"P90 outage duration    : {risk['p90_finish']:.1f} h")
print(f"Schedule overrun risk  : {overrun_prob*100:.1f}% (prob > baseline)")


### 4a · Outage finish time distribution

In [ ]:
from outage_uncertainty.schedule_risk.monte_carlo import MonteCarloSimulator

# Re-run simulator directly to get the raw cp_times list for the histogram
simulator = MonteCarloSimulator(network, n_samples=2000)
sim_result = simulator.run()
cp_times   = sim_result.cp_times
overrun_prob = 1.0 - risk.get("robustness", 0.0)

fig, ax = plt.subplots(figsize=(10, 4.5))

ax.hist(cp_times, bins=50, color="#4C72B0", alpha=0.75,
        edgecolor="white", linewidth=0.4, density=True, label="Simulated finish")

ax.axvline(baseline_cp,          color="#2c3e50", lw=2,   ls="-",
           label=f"Baseline  {baseline_cp:.0f} h")
ax.axvline(risk["p50_finish"],   color="#27ae60", lw=2,   ls="--",
           label=f"P50  {risk['p50_finish']:.1f} h")
ax.axvline(risk["p80_finish"],   color="#e67e22", lw=1.8, ls="--",
           label=f"P80  {risk['p80_finish']:.1f} h")
ax.axvline(risk["p90_finish"],   color="#e74c3c", lw=1.8, ls=":",
           label=f"P90  {risk['p90_finish']:.1f} h")

overrun = [v for v in cp_times if v > baseline_cp]
if overrun:
    ax.hist(overrun, bins=50, color="#e74c3c", alpha=0.25, edgecolor="none", density=True)

ax.set_xlabel("Outage duration (h)", fontsize=11)
ax.set_ylabel("Probability density", fontsize=11)
ax.set_title(
    f"OH-005 finish time distribution — 2 000 MC iterations\n"
    f"Schedule overrun risk: {overrun_prob*100:.1f}%",
    fontsize=11
)
ax.legend(fontsize=9)
ax.yaxis.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 4b · Activity risk ranking — criticality, drag, and CP sensitivity

In [ ]:
ci  = risk["criticality_index"]   # fraction of iterations on CP
drg = risk["expected_drag"]       # expected time added when critical
sen = risk["cp_sensitivity"]      # point-biserial correlation with overrun

# Only include the planned work activities
ranked_ids = sorted(
    [k for k in ci if k.startswith("Q-") and k not in ("Q-INIT","Q-END","Q-REPL")],
    key=lambda k: sen.get(k, 0),
    reverse=True
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
palette = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, len(ranked_ids)))

# Criticality index
crit_vals = [ci.get(k, 0) * 100 for k in ranked_ids]
axes[0].barh(ranked_ids, crit_vals, color=palette)
axes[0].set_xlabel("Criticality index (%)")
axes[0].set_title("% of simulations on CP")
axes[0].invert_yaxis()
for i, v in enumerate(crit_vals):
    axes[0].text(v + 0.5, i, f"{v:.0f}%", va="center", fontsize=8)

# Expected drag
drag_vals = [drg.get(k, 0) for k in ranked_ids]
axes[1].barh(ranked_ids, drag_vals, color=palette)
axes[1].set_xlabel("Expected drag (h)")
axes[1].set_title("Avg hours added when on CP")
axes[1].invert_yaxis()
for i, v in enumerate(drag_vals):
    axes[1].text(v + 0.05, i, f"{v:.1f}h", va="center", fontsize=8)

# CP sensitivity
sen_vals = [sen.get(k, 0) for k in ranked_ids]
axes[2].barh(ranked_ids, sen_vals, color=palette)
axes[2].set_xlabel("CP sensitivity (correlation)")
axes[2].set_title("Schedule leverage")
axes[2].invert_yaxis()
for i, v in enumerate(sen_vals):
    axes[2].text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=8)

fig.suptitle("OH-005 Activity Risk Ranking (sorted by CP sensitivity)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


### 4c · CP path frequency heatmap

In [ ]:
from collections import Counter

path_counts = Counter(tuple(p) for p in sim_result.cp_paths)
top_paths   = path_counts.most_common(6)
n_runs      = len(sim_result.cp_paths)

all_acts = sorted({a for path, _ in top_paths for a in path})
matrix = np.zeros((len(top_paths), len(all_acts)))
for i, (path, _) in enumerate(top_paths):
    for j, act in enumerate(all_acts):
        matrix[i, j] = 1.0 if act in path else 0.0

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.imshow(matrix, aspect="auto", cmap="Blues", vmin=0, vmax=1)

path_labels = [f"{cnt/n_runs*100:.0f}%" for _, cnt in top_paths]
ax.set_yticks(range(len(top_paths)))
ax.set_yticklabels(path_labels)
ax.set_xticks(range(len(all_acts)))
ax.set_xticklabels(all_acts, rotation=15, ha="right")
ax.set_ylabel("Path frequency")
ax.set_title("Top-6 critical path routes across 2 000 simulations", fontsize=11)

for i in range(len(top_paths)):
    for j in range(len(all_acts)):
        if matrix[i, j] > 0:
            ax.text(j, i, "\u25cf", ha="center", va="center", color="#2c3e50", fontsize=14)

plt.tight_layout()
plt.show()


---
## Summary

| Step | Component | Key output |
|------|-----------|------------|
| Pre-processing | `AbbreviationResolver` + `DomainSpellChecker` | Cleaned description |
| Retrieval | `SimilarityEngine` (lexical + semantic + context) | Top-k analogues + scores |
| Estimation | `OutlierHandler` + `DistributionFitter` | P50 / P80 / P90 per activity |
| Risk | `MonteCarloSimulator` + `CriticalPathRiskAnalyzer` | Outage P90, criticality index, drag |

**To enable Ollama-based semantic embeddings** (higher retrieval quality):
```python
from outage_uncertainty.api.config import AppConfig
service = build_duration_uncertainty_service(AppConfig(
    embedding_enabled=True,
    embedding_model="nomic-embed-text:latest",
    llm_disambiguation_enabled=True,
    llm_model="mistral:latest",
))
```